# Unstain → H&E hybrid training

이 노트북은 다음 문제를 줄이도록 구성했습니다.

- 밝고 대비가 약한 Unstain 입력: grayscale optical-density(OD) 정규화
- 미세 정합 오차에 대한 과도한 paired loss 의존: 128/256/blurred-512 다중 해상도 구조 손실
- 그럴듯한 H&E 분포 학습: paired 입력을 보지 않는 PatchGAN discriminator
- 구조 보존: dropout 없는 U-Net skip connection
- hallucination 억제: Unstain OD edge와 생성 H&E OD edge의 위치 상관 손실

생성 방향은 명확히 `Unstain (1 channel) → H&E (3 channels)`입니다.

In [1]:
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = torch.cuda.is_available()
print('device:', device)

device: cuda:1


/home/user/anaconda3/envs/urban/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
params = {
    'data_dir': Path('../../data/HnE_n_UNStaining/patch_dataset_mpp05_2048'),
    'output_dir': Path('../../results/Unstain2HnE_hybrid_v3'),
    'checkpoint_dir': Path('../../model/Unstain2HnE_hybrid_v3'),
    'image_ext': 'png',
    'image_max_count': 30000,
    'original_size': 2048,
    'input_size': 512,
    'mpp_scales': (0.5,),  # train and validation use the same physical scale
    'batch_size': 4,
    'num_epochs': 200,
    'val_fraction': 0.10,
    'preload_images': True,
    'max_cache_gib': 64,
    'od_background_threshold': 0.98,
    'od_quantile': 0.995,
    'od_calibration_images': 256,
    'min_crop_tissue_fraction': 0.10,
    'crop_retry_count': 10,
    'tissue_check_size': 64,
    'lowres_size': 128,
    'midres_size': 256,
    'fullres_blur_kernel': 9,
    'fullres_blur_sigma': 2.0,
    'base_channels': 48,
    'lr_g': 2e-4,
    'lr_d': 2e-4,
    'beta1': 0.5,
    'beta2': 0.999,
    'lambda_gan': 0.05,
    'gan_warmup_epochs': 30,
    'gan_ramp_epochs': 20,
    'lambda_lowres': 5.0,
    'lambda_ssim': 2.0,
    'lambda_gradient': 0.5,
    'lambda_fullres_blur': 2.0,
    'lambda_input_structure': 0.5,
    'save_every': 10,
    'resume_checkpoint': None,
}

params['output_dir'].mkdir(parents=True, exist_ok=True)
params['checkpoint_dir'].mkdir(parents=True, exist_ok=True)

## Dataset and deterministic Unstain normalization

OD 상한은 학습 영상 표본에서 한 번 계산한 뒤 모든 패치에 고정 적용합니다. 패치별 min-max나 CLAHE는 빈 배경 노이즈를 증폭할 수 있어 사용하지 않습니다. Train과 validation은 모두 0.5 MPP를 사용합니다. 학습 crop은 tissue fraction을 검사하고, validation은 중앙부터 고정 순서로 tissue 기준을 만족하는 첫 crop을 선택합니다.

In [3]:
def slide_key(path):
    return re.sub(r'_\d{5}_x-?\d+_y-?\d+$', '', Path(path).stem)


def collect_pairs(data_dir, image_ext='png'):
    hne_paths = sorted((data_dir / 'hne').glob(f'*.{image_ext}'))
    unstain_by_name = {p.name: p for p in (data_dir / 'unstain').glob(f'*.{image_ext}')}
    missing = [p.name for p in hne_paths if p.name not in unstain_by_name]
    if missing:
        raise FileNotFoundError(f'Missing Unstain pair for {missing[0]}')
    pairs = [(unstain_by_name[p.name], p) for p in hne_paths]
    if not pairs:
        raise RuntimeError(f'No paired images found under {data_dir}')
    return pairs


def split_pairs_by_slide(pairs, val_fraction=0.1, seed=42):
    groups = {}
    for pair in pairs:
        groups.setdefault(slide_key(pair[0]), []).append(pair)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    if len(keys) == 1:
        shuffled = pairs.copy()
        random.Random(seed).shuffle(shuffled)
        n_val = max(1, round(len(shuffled) * val_fraction))
        return shuffled[n_val:], shuffled[:n_val]
    n_val = min(len(keys) - 1, max(1, round(len(keys) * val_fraction)))
    val_keys = set(keys[:n_val])
    train_pairs = [pair for key in keys if key not in val_keys for pair in groups[key]]
    val_pairs = [pair for key in keys if key in val_keys for pair in groups[key]]
    return train_pairs, val_pairs


class PairedUnstainHnEDataset(Dataset):
    def __init__(self, pairs, params, training=True, od_max=None):
        self.pairs = list(pairs)
        self.training = training
        self.original_size = params['original_size']
        self.output_size = params['input_size']
        self.mpp_scales = tuple(params['mpp_scales']) if training else (0.5,)
        self.preload_images = params['preload_images']
        self.max_cache_gib = params['max_cache_gib']
        self.od_background_threshold = params['od_background_threshold']
        self.od_quantile = params['od_quantile']
        self.od_calibration_images = params['od_calibration_images']
        self.min_crop_tissue_fraction = params['min_crop_tissue_fraction']
        self.crop_retry_count = params['crop_retry_count']
        self.tissue_check_size = params['tissue_check_size']
        self.unstain_images = None
        self.hne_images = None

        if self.preload_images:
            self._preload_to_ram()
        self.od_max = float(od_max) if od_max is not None else self._estimate_od_max()
        print(f'{"train" if training else "val"}: {len(self):,} pairs, fixed OD_MAX={self.od_max:.4f}')

    def __len__(self):
        return len(self.pairs)

    def _read_rgb(self, path):
        with Image.open(path) as image:
            image = image.convert('RGB')
        if image.size != (self.original_size, self.original_size):
            image = TF.resize(
                image, [self.original_size, self.original_size],
                interpolation=InterpolationMode.BILINEAR, antialias=True
            )
        return image

    def _preload_to_ram(self):
        estimated_gib = len(self) * 2 * self.original_size**2 * 3 / 1024**3
        if estimated_gib > self.max_cache_gib:
            raise MemoryError(
                f'Estimated decoded cache {estimated_gib:.1f} GiB exceeds '
                f'max_cache_gib={self.max_cache_gib}.'
            )
        self.unstain_images, self.hne_images = [], []
        cache_bytes = 0
        for unstain_path, hne_path in tqdm(self.pairs, desc='Preloading RGB originals into RAM'):
            unstain = self._read_rgb(unstain_path)
            hne = self._read_rgb(hne_path)
            self.unstain_images.append(unstain)
            self.hne_images.append(hne)
            cache_bytes += (unstain.width * unstain.height + hne.width * hne.height) * 3
        print(f'RAM cache: {cache_bytes / 1024**3:.2f} GiB decoded RGB')

    def _get_originals(self, index):
        if self.preload_images:
            return self.unstain_images[index], self.hne_images[index]
        unstain_path, hne_path = self.pairs[index]
        return self._read_rgb(unstain_path), self._read_rgb(hne_path)

    def _estimate_od_max(self):
        count = min(len(self), self.od_calibration_images)
        indices = np.linspace(0, len(self) - 1, count, dtype=int)
        od_values = []
        for index in tqdm(indices, desc='Estimating fixed OD range'):
            unstain, _ = self._get_originals(int(index))
            small = unstain.resize((128, 128), Image.Resampling.BILINEAR).convert('L')
            gray = np.asarray(small, dtype=np.float32) / 255.0
            tissue = gray < self.od_background_threshold
            if tissue.any():
                od_values.append(-np.log(np.clip(gray[tissue], 1 / 255, 1.0)))
        if not od_values:
            raise RuntimeError('Could not estimate OD_MAX: no non-background pixels found.')
        od_max = np.quantile(np.concatenate(od_values), self.od_quantile)
        return max(float(od_max), 0.05)

    def _crop_tissue_fraction(self, unstain, top, left, crop_size):
        crop = unstain.crop((left, top, left + crop_size, top + crop_size))
        crop = crop.resize(
            (self.tissue_check_size, self.tissue_check_size), Image.Resampling.BILINEAR
        ).convert('L')
        gray = np.asarray(crop, dtype=np.float32) / 255.0
        return float(np.mean(gray < self.od_background_threshold))

    def _select_crop_coordinates(self, unstain, crop_size):
        limit = self.original_size - crop_size
        if limit <= 0:
            return 0, 0

        if self.training:
            candidates = [
                (random.randint(0, limit), random.randint(0, limit))
                for _ in range(self.crop_retry_count)
            ]
        else:
            # Deterministic validation: center + four corners.
            candidates = [
                (limit // 2, limit // 2), (0, 0), (0, limit),
                (limit, 0), (limit, limit),
            ]

        best_coordinates = candidates[0]
        best_fraction = -1.0
        for top, left in candidates:
            fraction = self._crop_tissue_fraction(unstain, top, left, crop_size)
            if fraction > best_fraction:
                best_fraction = fraction
                best_coordinates = (top, left)
            if fraction >= self.min_crop_tissue_fraction:
                return top, left
        return best_coordinates

    def _paired_view(self, unstain, hne):
        selected_mpp = random.choice(self.mpp_scales)
        crop_size = min(self.original_size, round(self.output_size * selected_mpp / 0.5))
        top, left = self._select_crop_coordinates(unstain, crop_size)
        unstain = TF.crop(unstain, top, left, crop_size, crop_size)
        hne = TF.crop(hne, top, left, crop_size, crop_size)
        if crop_size != self.output_size:
            unstain = TF.resize(unstain, [self.output_size] * 2, InterpolationMode.BILINEAR, antialias=True)
            hne = TF.resize(hne, [self.output_size] * 2, InterpolationMode.BILINEAR, antialias=True)
        if self.training:
            angle = random.choice((0, 90, 180, 270))
            if angle:
                unstain = TF.rotate(unstain, angle)
                hne = TF.rotate(hne, angle)
            if random.random() < 0.5:
                unstain, hne = TF.hflip(unstain), TF.hflip(hne)
            if random.random() < 0.5:
                unstain, hne = TF.vflip(unstain), TF.vflip(hne)
        return unstain, hne

    def _unstain_od_tensor(self, image):
        rgb = TF.to_tensor(image)
        gray = TF.rgb_to_grayscale(rgb, num_output_channels=1)
        od = -torch.log(gray.clamp(1 / 255, 1.0))
        od = (od / self.od_max).clamp(0, 1)
        return od * 2 - 1

    def __getitem__(self, index):
        unstain, hne = self._get_originals(index)
        unstain, hne = self._paired_view(unstain, hne)
        unstain = self._unstain_od_tensor(unstain)
        hne = TF.to_tensor(hne) * 2 - 1
        return unstain, hne

In [ ]:
all_pairs = collect_pairs(params['data_dir'], params['image_ext'])
if len(all_pairs) > params['image_max_count']:
    all_pairs = random.Random(SEED).sample(all_pairs, params['image_max_count'])
train_pairs, val_pairs = split_pairs_by_slide(all_pairs, params['val_fraction'], SEED)
print(f'all={len(all_pairs):,}, train={len(train_pairs):,}, val={len(val_pairs):,}')
print(f'train slides={len({slide_key(p[0]) for p in train_pairs})}, val slides={len({slide_key(p[0]) for p in val_pairs})}')

train_dataset = PairedUnstainHnEDataset(train_pairs, params, training=True)
val_dataset = PairedUnstainHnEDataset(val_pairs, params, training=False, od_max=train_dataset.od_max)

train_loader = DataLoader(
    train_dataset, batch_size=params['batch_size'], shuffle=True, num_workers=0,
    pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset, batch_size=params['batch_size'], shuffle=False, num_workers=0,
    pin_memory=torch.cuda.is_available()
)

all=1,052, train=904, val=148
train slides=73, val slides=8


Preloading RGB originals into RAM:   0%|          | 0/904 [00:00<?, ?it/s]

In [ ]:
unstain_batch, hne_batch = next(iter(train_loader))
fig, axes = plt.subplots(2, min(4, len(unstain_batch)), figsize=(14, 7))
for i in range(axes.shape[1]):
    axes[0, i].imshow(((unstain_batch[i, 0] + 1) / 2).numpy(), cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title('Unstain OD input')
    axes[1, i].imshow(((hne_batch[i] + 1) / 2).permute(1, 2, 0).numpy().clip(0, 1))
    axes[1, i].set_title('Registered H&E target')
    axes[0, i].axis('off')
    axes[1, i].axis('off')
plt.tight_layout()

## Models

Generator는 1채널 OD 입력을 3채널 H&E로 변환합니다. 모든 decoder는 bilinear resize + reflection-padded convolution을 사용해 transposed-convolution checkerboard를 방지합니다. Discriminator에는 Unstain 입력을 주지 않아 작은 registration error를 직접 판별 근거로 사용하지 않게 합니다.

In [ ]:
class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=not normalize)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_channels, out_channels, 3, 1, 0, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UnstainToHnEGenerator(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1 = DownBlock(1, base, normalize=False)
        self.e2 = DownBlock(base, base * 2)
        self.e3 = DownBlock(base * 2, base * 4)
        self.e4 = DownBlock(base * 4, base * 8)
        self.bottleneck = DownBlock(base * 8, base * 16)
        self.u4 = UpBlock(base * 16, base * 8)
        self.u3 = UpBlock(base * 16, base * 4)
        self.u2 = UpBlock(base * 8, base * 2)
        self.u1 = UpBlock(base * 4, base)
        self.output = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.ReflectionPad2d(1),
            nn.Conv2d(base * 2, 3, 3, 1, 0),
            nn.Tanh(),
        )

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(e1)
        e3 = self.e3(e2)
        e4 = self.e4(e3)
        b = self.bottleneck(e4)
        u4 = self.u4(b)
        u3 = self.u3(torch.cat([u4, e4], dim=1))
        u2 = self.u2(torch.cat([u3, e3], dim=1))
        u1 = self.u1(torch.cat([u2, e2], dim=1))
        return self.output(torch.cat([u1, e1], dim=1))


class HnEPatchDiscriminator(nn.Module):
    def __init__(self, base=64):
        super().__init__()
        def block(in_channels, out_channels, normalize=True):
            layers = [nn.Conv2d(in_channels, out_channels, 4, 2, 1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_channels))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        self.model = nn.Sequential(
            *block(3, base, normalize=False),
            *block(base, base * 2),
            *block(base * 2, base * 4),
            *block(base * 4, base * 8),
            nn.Conv2d(base * 8, 1, 4, 1, 1),
        )

    def forward(self, x):
        return self.model(x)


def init_weights(module):
    if isinstance(module, nn.Conv2d):
        nn.init.normal_(module.weight, 0.0, 0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


G = UnstainToHnEGenerator(params['base_channels']).to(device)
D = HnEPatchDiscriminator().to(device)
G.apply(init_weights)
D.apply(init_weights)
print(f'G parameters: {sum(p.numel() for p in G.parameters()) / 1e6:.2f} M')
print(f'D parameters: {sum(p.numel() for p in D.parameters()) / 1e6:.2f} M')

## Registration-tolerant hybrid losses

128 px Charbonnier는 전체 색과 배치를, 256 px SSIM/gradient는 중간 구조를, Gaussian-blurred 512 px Charbonnier는 고해상도 구조를 감독합니다. Unstain OD와 생성 H&E OD의 Sobel edge correlation은 입력에 없는 구조 생성을 억제합니다. 첫 30 epoch는 구조 손실만 학습하고 이후 20 epoch 동안 GAN 가중치를 0.05까지 올립니다.

In [ ]:
def to_01(x):
    return (x + 1) / 2


def comparison_view(x, size):
    return F.interpolate(to_01(x), size=(size, size), mode='area')


def charbonnier_loss(x, y, eps=1e-3):
    return torch.sqrt((x - y).square() + eps**2).mean()


def ssim_index(x, y, window_size=11):
    padding = window_size // 2
    mu_x = F.avg_pool2d(x, window_size, 1, padding)
    mu_y = F.avg_pool2d(y, window_size, 1, padding)
    sigma_x = F.avg_pool2d(x * x, window_size, 1, padding) - mu_x.square()
    sigma_y = F.avg_pool2d(y * y, window_size, 1, padding) - mu_y.square()
    sigma_xy = F.avg_pool2d(x * y, window_size, 1, padding) - mu_x * mu_y
    c1, c2 = 0.01**2, 0.03**2
    score = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / (
        (mu_x.square() + mu_y.square() + c1) * (sigma_x + sigma_y + c2)
    )
    return score.mean()


def rgb_luminance(x):
    weights = x.new_tensor([0.2126, 0.7152, 0.0722]).view(1, 3, 1, 1)
    return (x * weights).sum(dim=1, keepdim=True)


def gradient_loss(x, y):
    x, y = rgb_luminance(x), rgb_luminance(y)
    x_dx, y_dx = x[:, :, :, 1:] - x[:, :, :, :-1], y[:, :, :, 1:] - y[:, :, :, :-1]
    x_dy, y_dy = x[:, :, 1:, :] - x[:, :, :-1, :], y[:, :, 1:, :] - y[:, :, :-1, :]
    return charbonnier_loss(x_dx, y_dx) + charbonnier_loss(x_dy, y_dy)


def sobel_magnitude(x):
    kernel_x = x.new_tensor([
        [-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]
    ]).view(1, 1, 3, 3) / 8.0
    kernel_y = kernel_x.transpose(2, 3)
    x = F.pad(x, (1, 1, 1, 1), mode='reflect')
    dx = F.conv2d(x, kernel_x)
    dy = F.conv2d(x, kernel_y)
    return torch.sqrt(dx.square() + dy.square() + 1e-6)


def input_structure_loss(fake_hne, unstain_od):
    size = params['midres_size']
    input_od = F.interpolate(to_01(unstain_od), size=(size, size), mode='area')
    fake_gray = rgb_luminance(to_01(fake_hne)).clamp(1 / 255, 1.0)
    fake_od = -torch.log(fake_gray)
    fake_od = F.interpolate(fake_od, size=(size, size), mode='area')
    input_edges = sobel_magnitude(input_od).flatten(1)
    fake_edges = sobel_magnitude(fake_od).flatten(1)
    correlation = F.cosine_similarity(input_edges, fake_edges, dim=1, eps=1e-6)
    return 1 - correlation.mean()


def structural_losses(fake_hne, real_hne, unstain_od):
    fake_01, real_01 = to_01(fake_hne), to_01(real_hne)
    fake_low = F.interpolate(
        fake_01, size=(params['lowres_size'],) * 2, mode='area'
    )
    real_low = F.interpolate(
        real_01, size=(params['lowres_size'],) * 2, mode='area'
    )
    fake_mid = F.interpolate(
        fake_01, size=(params['midres_size'],) * 2, mode='area'
    )
    real_mid = F.interpolate(
        real_01, size=(params['midres_size'],) * 2, mode='area'
    )
    kernel = [params['fullres_blur_kernel']] * 2
    sigma = [params['fullres_blur_sigma']] * 2
    fake_blur = TF.gaussian_blur(fake_01, kernel_size=kernel, sigma=sigma)
    real_blur = TF.gaussian_blur(real_01, kernel_size=kernel, sigma=sigma)

    lowres = charbonnier_loss(fake_low, real_low)
    ssim_loss = 1 - ssim_index(fake_mid, real_mid)
    gradients = gradient_loss(fake_mid, real_mid)
    fullres_blur = charbonnier_loss(fake_blur, real_blur)
    input_structure = input_structure_loss(fake_hne, unstain_od)
    structural_total = (
        params['lambda_lowres'] * lowres
        + params['lambda_ssim'] * ssim_loss
        + params['lambda_gradient'] * gradients
        + params['lambda_fullres_blur'] * fullres_blur
        + params['lambda_input_structure'] * input_structure
    )
    return {
        'lowres': lowres,
        'ssim_loss': ssim_loss,
        'ssim_score': 1 - ssim_loss,
        'gradient': gradients,
        'fullres_blur': fullres_blur,
        'input_structure': input_structure,
        'structural_total': structural_total,
    }


def scheduled_gan_weight(epoch):
    if epoch < params['gan_warmup_epochs']:
        return 0.0
    ramp_epoch = epoch - params['gan_warmup_epochs'] + 1
    ramp = min(1.0, ramp_epoch / max(params['gan_ramp_epochs'], 1))
    return params['lambda_gan'] * ramp


criterion_gan = nn.MSELoss()

In [ ]:
optimizer_g = torch.optim.Adam(
    G.parameters(), lr=params['lr_g'], betas=(params['beta1'], params['beta2'])
)
optimizer_d = torch.optim.Adam(
    D.parameters(), lr=params['lr_d'], betas=(params['beta1'], params['beta2'])
)
scheduler_g = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_g, params['num_epochs'])
scheduler_d = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_d, params['num_epochs'])
amp_enabled = device.type == 'cuda'
scaler_g = torch.amp.GradScaler(device=device, enabled=amp_enabled)
scaler_d = torch.amp.GradScaler(device=device, enabled=amp_enabled)
start_epoch = 0
history = []
best_val_score = float('inf')
best_epoch = -1

if params['resume_checkpoint'] is not None:
    checkpoint = torch.load(params['resume_checkpoint'], map_location=device)
    G.load_state_dict(checkpoint['G'])
    D.load_state_dict(checkpoint['D'])
    optimizer_g.load_state_dict(checkpoint['optimizer_g'])
    optimizer_d.load_state_dict(checkpoint['optimizer_d'])
    scheduler_g.load_state_dict(checkpoint['scheduler_g'])
    scheduler_d.load_state_dict(checkpoint['scheduler_d'])
    scaler_g.load_state_dict(checkpoint['scaler_g'])
    scaler_d.load_state_dict(checkpoint['scaler_d'])
    history = checkpoint.get('history', [])
    best_val_score = checkpoint.get('best_val_score', float('inf'))
    best_epoch = checkpoint.get('best_epoch', -1)
    start_epoch = checkpoint['epoch'] + 1
    print('resumed from epoch', start_epoch)

In [ ]:
def set_requires_grad(model, enabled):
    for parameter in model.parameters():
        parameter.requires_grad_(enabled)


def save_preview(epoch, unstain, real_hne, fake_hne):
    count = min(3, len(unstain))
    fig, axes = plt.subplots(count, 4, figsize=(14, 4 * count), squeeze=False)
    try:
        for i in range(count):
            # AMP output can be float16/bfloat16, which Matplotlib cannot render.
            u = to_01(unstain[i]).detach().float().cpu()[0].numpy()
            real = (
                to_01(real_hne[i]).detach().float().cpu()
                .permute(1, 2, 0).clamp(0, 1).numpy()
            )
            fake = (
                to_01(fake_hne[i]).detach().float().cpu()
                .permute(1, 2, 0).clamp(0, 1).numpy()
            )
            error = np.abs(fake - real).mean(axis=2).astype(np.float32, copy=False)
            axes[i, 0].imshow(u, cmap='gray', vmin=0, vmax=1)
            axes[i, 0].set_title('Unstain OD input')
            axes[i, 1].imshow(real)
            axes[i, 1].set_title('Real H&E')
            axes[i, 2].imshow(fake)
            axes[i, 2].set_title('Generated H&E')
            axes[i, 3].imshow(error, cmap='magma', vmin=0, vmax=0.5)
            axes[i, 3].set_title('Absolute error')
            for axis in axes[i]:
                axis.axis('off')
        plt.tight_layout()
        fig.savefig(params['output_dir'] / f'epoch_{epoch:04d}.png', dpi=160, bbox_inches='tight')
    finally:
        plt.close(fig)


@torch.no_grad()
def validate(max_batches=None):
    G.eval()
    metric_names = (
        'lowres', 'ssim_loss', 'ssim_score', 'gradient',
        'fullres_blur', 'input_structure', 'structural_total',
    )
    totals = {name: 0.0 for name in metric_names}
    count = 0
    last_batch = None
    for unstain, real_hne in val_loader:
        unstain = unstain.to(device, non_blocking=True)
        real_hne = real_hne.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
            fake_hne = G(unstain)
            losses = structural_losses(fake_hne, real_hne, unstain)
            for name in metric_names:
                totals[name] += losses[name].item()
        count += 1
        last_batch = (unstain, real_hne, fake_hne)
        if max_batches is not None and count >= max_batches:
            break
    return {key: value / max(count, 1) for key, value in totals.items()}, last_batch

In [ ]:
history = globals().get('history', [])  # keep completed epochs if this cell is rerun
for epoch in range(start_epoch, params['num_epochs']):
    G.train()
    D.train()
    gan_weight = scheduled_gan_weight(epoch)
    metric_names = (
        'G', 'D', 'gan', 'lowres', 'ssim_loss', 'ssim_score',
        'gradient', 'fullres_blur', 'input_structure', 'structural_total',
    )
    totals = {name: 0.0 for name in metric_names}
    progress = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{params["num_epochs"]}')

    for step, (unstain, real_hne) in enumerate(progress, start=1):
        unstain = unstain.to(device, non_blocking=True)
        real_hne = real_hne.to(device, non_blocking=True)

        set_requires_grad(D, False)
        optimizer_g.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
            fake_hne = G(unstain)
            structure = structural_losses(fake_hne, real_hne, unstain)
            if gan_weight > 0:
                pred_fake = D(fake_hne)
                loss_gan = criterion_gan(pred_fake, torch.ones_like(pred_fake))
            else:
                loss_gan = fake_hne.new_zeros(())
            loss_g = structure['structural_total'] + gan_weight * loss_gan
        scaler_g.scale(loss_g).backward()
        scaler_g.step(optimizer_g)
        scaler_g.update()

        if gan_weight > 0:
            set_requires_grad(D, True)
            optimizer_d.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                pred_real = D(real_hne)
                pred_fake = D(fake_hne.detach())
                loss_d_real = criterion_gan(pred_real, torch.ones_like(pred_real))
                loss_d_fake = criterion_gan(pred_fake, torch.zeros_like(pred_fake))
                loss_d = 0.5 * (loss_d_real + loss_d_fake)
            scaler_d.scale(loss_d).backward()
            scaler_d.step(optimizer_d)
            scaler_d.update()
        else:
            loss_d = fake_hne.new_zeros(())

        current = {
            'G': loss_g.item(), 'D': loss_d.item(), 'gan': loss_gan.item(),
            **{name: value.item() for name, value in structure.items()},
        }
        for name, value in current.items():
            totals[name] += value
        progress.set_postfix(
            G=f'{totals["G"] / step:.3f}',
            D=f'{totals["D"] / step:.3f}',
            SSIM=f'{totals["ssim_score"] / step:.3f}',
            GAN_W=f'{gan_weight:.2f}',
        )

    scheduler_g.step()
    if gan_weight > 0:
        scheduler_d.step()
    train_metrics = {name: value / max(len(train_loader), 1) for name, value in totals.items()}
    train_metrics['gan_weight'] = gan_weight
    val_metrics, preview_batch = validate()
    history.append({'epoch': epoch, 'train': train_metrics, 'val': val_metrics})
    print('train:', {k: round(v, 4) for k, v in train_metrics.items()})
    print('val:', {k: round(v, 4) for k, v in val_metrics.items()})

    is_best = val_metrics['structural_total'] < best_val_score
    if is_best:
        best_val_score = val_metrics['structural_total']
        best_epoch = epoch

    checkpoint = {
        'epoch': epoch, 'G': G.state_dict(), 'D': D.state_dict(),
        'optimizer_g': optimizer_g.state_dict(), 'optimizer_d': optimizer_d.state_dict(),
        'scheduler_g': scheduler_g.state_dict(), 'scheduler_d': scheduler_d.state_dict(),
        'scaler_g': scaler_g.state_dict(), 'scaler_d': scaler_d.state_dict(),
        'od_max': train_dataset.od_max, 'params': params, 'history': history,
        'best_val_score': best_val_score, 'best_epoch': best_epoch,
    }
    torch.save(checkpoint, params['checkpoint_dir'] / 'latest.pt')
    if is_best:
        torch.save(checkpoint, params['checkpoint_dir'] / 'best.pt')
        print(f'new best: epoch {epoch + 1}, val structural={best_val_score:.4f}')
    if (epoch + 1) % params['save_every'] == 0:
        torch.save(checkpoint, params['checkpoint_dir'] / f'epoch_{epoch + 1:04d}.pt')

    if preview_batch is not None:
        save_preview(epoch, *preview_batch)